# CareBridge AI server on a free Colab T4

Runs the AI the CareBridge backend needs on a free Colab T4 GPU:
- **Speech-to-text:** Whisper, used for call minutes.
- **Language model:** Ollama, used for Tier 3 autocomplete and the minutes gap check.

Both sit behind **one token-protected ngrok address**.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. In the 🔑 **Secrets** panel (left sidebar), add the following and turn on notebook access for each:
   - `NGROK_AUTHTOKEN`: from https://dashboard.ngrok.com/get-started/your-authtoken
   - `NGROK_DOMAIN` (optional but recommended): your free static domain from https://dashboard.ngrok.com/domains. With it, the address stays the same every session.
   - `GATEWAY_TOKEN`: any long random string. If it is missing, one is generated for you.
3. *Runtime → Run all*. The first run takes about 5–8 minutes for downloads. At the end it prints the lines to paste into CareBridge's `.env`.

⚠️ Use **test or demo conversations only** on Colab. Google runs the GPU and ngrok carries the traffic, so this setup is not the sovereign production deployment.

## 1 · Settings

In [ ]:
WHISPER_MODEL = "large-v3"          # best for Hindi/Kannada; "medium" is faster
LLM_MODEL = "qwen2.5:7b-instruct"   # fits next to Whisper on a T4; "qwen2.5:3b-instruct" is faster
GATEWAY_PORT = 8765                 # avoid 8080, which Colab itself may use

## 2 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3 · Install Ollama and the gateway's Python packages

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
# av is pinned: PyAV 19 removed an option faster-whisper uses. The nvidia-* packages provide the CUDA 12
# libraries Whisper's engine (CTranslate2) needs, since newer Colab images ship a different CUDA version.
!pip -q install faster-whisper "av==18.1.0" "nvidia-cublas-cu12" "nvidia-cudnn-cu12==9.*" fastapi uvicorn httpx python-multipart pyngrok

## 4 · Start Ollama and download the language model

In [ ]:
import os, subprocess, time, httpx

subprocess.run(["pkill", "-f", "ollama serve"]); time.sleep(1)   # safe to re-run this cell
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="-1", OLLAMA_NUM_PARALLEL="2")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env,
                               stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        httpx.get("http://127.0.0.1:11434/api/version", timeout=2); break
    except httpx.HTTPError:
        time.sleep(1)
!ollama pull {LLM_MODEL}

## 5 · Write the gateway (speech-to-text + Ollama pass-through, token-protected)

In [ ]:
%%writefile /content/ai_gateway.py
"""CareBridge AI gateway: one token-protected endpoint for speech-to-text and the language model.

Runs next to the GPU (e.g. a Google Colab T4) and is exposed through a single tunnel.

  POST /v1/audio/transcriptions   faster-whisper, OpenAI-compatible (what SCRIBE_ASR_PROVIDER=http calls)
  POST /v1/audio/translations     same, translated to English
  *    /api/...                   passed through to the local Ollama server (Tier 3 + call minutes)
  GET  /health                    no token needed; reports nothing about patients

Nothing is written to disk and request bodies are never logged.

Environment:
  GATEWAY_TOKEN   required shared secret (the backend's AI_GATEWAY_TOKEN)
  WHISPER_MODEL   default "large-v3"
  WHISPER_DEVICE  "cuda" (default) or "cpu"
  OLLAMA_URL      default http://127.0.0.1:11434
"""
import asyncio
import hmac
import io
import os
import threading
import traceback

import httpx
from fastapi import FastAPI, File, Form, Request, UploadFile
from fastapi.responses import JSONResponse, StreamingResponse
from starlette.background import BackgroundTask

TOKEN = os.environ.get("GATEWAY_TOKEN", "")
if len(TOKEN) < 16:
    raise SystemExit("Set GATEWAY_TOKEN to a random secret of at least 16 characters.")

WHISPER_MODEL = os.environ.get("WHISPER_MODEL", "large-v3")
WHISPER_DEVICE = os.environ.get("WHISPER_DEVICE", "cuda")
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434").rstrip("/")
MAX_AUDIO_BYTES = 8 * 1024 * 1024

app = FastAPI(title="CareBridge AI gateway", docs_url=None, redoc_url=None, openapi_url=None)
_whisper = None
_whisper_error = ""                   # set when the startup self-test fails (e.g. missing CUDA libraries)
_whisper_lock = threading.Lock()      # one transcription at a time on the GPU
_ollama = httpx.AsyncClient(timeout=httpx.Timeout(300.0, connect=5.0))


def _load_whisper():
    global _whisper
    if _whisper is None:
        from faster_whisper import WhisperModel
        compute = "float16" if WHISPER_DEVICE == "cuda" else "int8"
        _whisper = WhisperModel(WHISPER_MODEL, device=WHISPER_DEVICE, compute_type=compute)
    return _whisper


def _self_test() -> None:
    """Run one second of silence through the full GPU + audio-decoding path, so problems show at startup."""
    global _whisper_error
    import wave
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000); w.writeframes(bytes(32000))
    try:
        segments, _ = _load_whisper().transcribe(io.BytesIO(buf.getvalue()), beam_size=1, language="en")
        list(segments)
        _whisper_error = ""
    except Exception as exc:
        traceback.print_exc()
        _whisper_error = f"{type(exc).__name__}: {str(exc)[:200]}"


@app.on_event("startup")
async def _startup() -> None:
    await asyncio.to_thread(_self_test)


@app.middleware("http")
async def _require_token(request: Request, call_next):
    if request.url.path != "/health":
        sent = request.headers.get("authorization", "")
        if not hmac.compare_digest(sent.encode(), f"Bearer {TOKEN}".encode()):
            return JSONResponse({"error": "unauthorized"}, status_code=401)
    return await call_next(request)


@app.get("/health")
async def health():
    try:
        ollama_ok = (await _ollama.get(f"{OLLAMA_URL}/api/version", timeout=3.0)).status_code == 200
    except httpx.HTTPError:
        ollama_ok = False
    return {"whisper": _whisper is not None and not _whisper_error, "whisper_model": WHISPER_MODEL,
            "whisper_error": _whisper_error or None, "ollama": ollama_ok}


def _run_whisper(audio: bytes, task: str, language: str | None):
    model = _load_whisper()
    with _whisper_lock:
        segments, info = model.transcribe(
            io.BytesIO(audio), task=task, language=language or None,
            beam_size=5, vad_filter=True, condition_on_previous_text=False,
        )
        segs = [{"start": s.start, "end": s.end, "text": s.text} for s in segments]
    return {
        "text": "".join(s["text"] for s in segs).strip(),
        "language": info.language,
        "duration": info.duration,
        "segments": segs,
    }


async def _audio_endpoint(task: str, file: UploadFile, language: str | None):
    audio = await file.read(MAX_AUDIO_BYTES + 1)
    if not audio:
        return JSONResponse({"error": "empty audio"}, status_code=400)
    if len(audio) > MAX_AUDIO_BYTES:
        return JSONResponse({"error": "audio too large"}, status_code=413)
    try:
        return await asyncio.to_thread(_run_whisper, audio, task, language)
    except Exception as exc:  # undecodable audio etc.; the traceback goes to gateway.log, never the audio
        traceback.print_exc()
        return JSONResponse({"error": f"transcription failed: {type(exc).__name__}: {str(exc)[:200]}"},
                            status_code=422)


@app.post("/v1/audio/transcriptions")
async def transcriptions(file: UploadFile = File(...), model: str = Form(""), language: str = Form(""),
                         response_format: str = Form("json")):
    return await _audio_endpoint("transcribe", file, language)


@app.post("/v1/audio/translations")
async def translations(file: UploadFile = File(...), model: str = Form(""), response_format: str = Form("json")):
    return await _audio_endpoint("translate", file, None)


@app.api_route("/api/{path:path}", methods=["GET", "POST", "DELETE"])
async def ollama_proxy(path: str, request: Request):
    """Pass-through to Ollama, streaming, so the backend can stop generation by closing the connection."""
    req = _ollama.build_request(
        request.method, f"{OLLAMA_URL}/api/{path}", content=await request.body(),
        headers={"content-type": request.headers.get("content-type", "application/json")},
    )
    try:
        upstream = await _ollama.send(req, stream=True)
    except httpx.HTTPError:
        return JSONResponse({"error": "ollama unavailable"}, status_code=502)
    return StreamingResponse(
        upstream.aiter_raw(), status_code=upstream.status_code,
        media_type=upstream.headers.get("content-type"), background=BackgroundTask(upstream.aclose),
    )

## 6 · Start the gateway (downloads the Whisper model on first run)

In [ ]:
import secrets
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return ""

import socket
subprocess.run(["pkill", "-f", "uvicorn ai_gateway:app"]); time.sleep(2)   # stop a gateway from an earlier run

def port_free(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) != 0

if not port_free(GATEWAY_PORT):                # something else holds the port: let the OS pick a free one
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0)); GATEWAY_PORT = s.getsockname()[1]
print("gateway port:", GATEWAY_PORT)
if not globals().get("GATEWAY_TOKEN"):
    GATEWAY_TOKEN = secret("GATEWAY_TOKEN") or secrets.token_urlsafe(32)
import nvidia.cublas.lib, nvidia.cudnn.lib
cuda12_libs = ":".join([list(nvidia.cublas.lib.__path__)[0], list(nvidia.cudnn.lib.__path__)[0]])
genv = dict(os.environ, GATEWAY_TOKEN=GATEWAY_TOKEN, WHISPER_MODEL=WHISPER_MODEL,
            WHISPER_DEVICE="cuda", OLLAMA_URL="http://127.0.0.1:11434",
            LD_LIBRARY_PATH=cuda12_libs + ":" + os.environ.get("LD_LIBRARY_PATH", ""))
gw_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "ai_gateway:app", "--host", "127.0.0.1", "--port", str(GATEWAY_PORT)],
    cwd="/content", env=genv, stdout=open("/content/gateway.log", "w"), stderr=subprocess.STDOUT)

for i in range(180):
    try:
        h = httpx.get(f"http://127.0.0.1:{GATEWAY_PORT}/health", timeout=2).json()
        print("gateway ready:", h)
        if not h["whisper"]:
            print("⚠️ Whisper failed its self-test; see /content/gateway.log")
        break
    except Exception:
        if gw_proc.poll() is not None:
            raise RuntimeError(open("/content/gateway.log").read()[-3000:])
        time.sleep(5)
else:
    print(open("/content/gateway.log").read()[-3000:])

## 7 · Warm up the language model

In [ ]:
r = httpx.post(f"http://127.0.0.1:{GATEWAY_PORT}/api/generate",
               headers={"Authorization": f"Bearer {GATEWAY_TOKEN}"},
               json={"model": LLM_MODEL, "prompt": "Patient complains of", "stream": False,
                     "options": {"num_predict": 12}}, timeout=300)
print(r.status_code, r.json().get("response"))

## 8 · Open the ngrok tunnel

In [ ]:
from pyngrok import ngrok, conf

ngrok.kill()   # close a tunnel from an earlier run, so the cell can be re-run
conf.get_default().auth_token = secret("NGROK_AUTHTOKEN")
domain = secret("NGROK_DOMAIN")
options = {"domain": domain} if domain else {}
tunnel = ngrok.connect(str(GATEWAY_PORT), "http", **options)
PUBLIC_URL = tunnel.public_url.replace("http://", "https://")

print("Public URL:", PUBLIC_URL)
print("Unauthenticated request is refused:", httpx.get(PUBLIC_URL + "/api/tags", timeout=20).status_code)

## 9 · Paste these lines into CareBridge's `.env`, then restart the backend

In [ ]:
print(f'''# --- Colab GPU ({time.strftime("%Y-%m-%d")}) ---
AI_GATEWAY_TOKEN={GATEWAY_TOKEN}
SCRIBE_ASR_PROVIDER=http
SCRIBE_ASR_URL={PUBLIC_URL}
SCRIBE_ASR_MODEL={WHISPER_MODEL}
SCRIBE_ASR_TRANSLATE=true
SCRIBE_ASR_TIMEOUT_S=90
SCRIBE_LLM_PROVIDER=local
SCRIBE_LLM_MODEL={LLM_MODEL}
INTELLISENSE_LLM_PROVIDER=local
INTELLISENSE_LOCAL_URL={PUBLIC_URL}
INTELLISENSE_LOCAL_MODEL={LLM_MODEL}
INTELLISENSE_LOCAL_KEEP_ALIVE=24h
INTELLISENSE_LLM_TIMEOUT_S=8
INTELLISENSE_HEALTH_INTERVAL_S=300''')
if not secret("GATEWAY_TOKEN"):
    print("\n# Tip: save this GATEWAY_TOKEN as a Colab secret so it stays the same next session.")

## 10 · Keep running (stop this cell to see logs; closing the tab ends the session)

In [ ]:
try:
    while True:
        h = httpx.get(f"http://127.0.0.1:{GATEWAY_PORT}/health", timeout=5).json()
        print(time.strftime("%H:%M:%S"), h, end="\r")
        time.sleep(60)
except KeyboardInterrupt:
    print("\n--- gateway.log (last lines) ---")
    print(open("/content/gateway.log").read()[-2000:])